统计各个instance的准确率correctness和spec的hook情况

In [ ]:
from pathlib import Path
import json
from collections import defaultdict
p=Path('logs/test_correct/gpt-oss-120b/gold')
wo_report_instance_ids=[]
def collect_test_correct_results(p,verbose=False):
    results=defaultdict(dict)
    for subdir in p.iterdir():
        if subdir.is_dir():
            instance_id=str(subdir.relative_to(p))
            
            report_p = subdir / "report.json"
            if not report_p.exists():
                results[instance_id]['pass']=False
                wo_report_instance_ids.append(instance_id)
                if verbose:
                    print(instance_id)
                continue
            report = json.loads(report_p.read_text())
            results[instance_id]['pass']=report[instance_id]['resolved']
                
            test_output_p = subdir / "test_output.txt"
            test_output=test_output_p.read_text()

            precondition_line = '!!!!!!!!!!!!!!!!!!!this is precondition!!!!!!!!!!!!!!!!!!!'
            lines = test_output.split('\n')
            pytest_condition = any(precondition_line in line and not line.strip().startswith('+') for line in lines)
            results[instance_id]['pytest'] = pytest_condition
    print(f'没有report：{len(wo_report_instance_ids)}')
    return results
results=collect_test_results(p,verbose=True)
len(wo_report_instance_ids)

统计各个instance的完备率completeness和spec的hook情况

In [ ]:
from pathlib import Path
import json
from collections import defaultdict
p=Path('logs/test_complete/QwQ-32B-thinking/gold')
wo_report_instance_ids=[]
def collect_test_complete_results(p,verbose=False):
    results=defaultdict(dict)
    wo_report_instance_ids_in=[]
    for subdir in p.iterdir():
        if subdir.is_dir():
            instance_id=str(subdir.relative_to(p))
            
            report_p = subdir / "report.json"
            if not report_p.exists():
                results[instance_id]['pass']=False
                wo_report_instance_ids_in.append(instance_id)
                wo_report_instance_ids.append(instance_id)
                if verbose:
                    print(instance_id)
                continue
            report = json.loads(report_p.read_text())
            results[instance_id]['pass']=report[instance_id]['tests_status']['FAIL_TO_PASS']['success']==[]
            test_output_p = subdir / "test_output.txt"
            test_output=test_output_p.read_text()

            lines = test_output.split('\n')
            precondition_line = 'precondition function failed'
            precondition_condition = any(precondition_line in line and not line.strip().startswith('+') for line in lines)
            
            postcondition_line = 'postcondition function failed'
            postcondition_condition = any(postcondition_line in line and not line.strip().startswith('+') for line in lines)

            results[instance_id]['pytest'] = precondition_condition or postcondition_condition
    print(f'没有report：{len(wo_report_instance_ids_in)}')
    return results
results=collect_test_results(p,verbose=False)
len(wo_report_instance_ids)

打印instance的各种通过情况

In [ ]:
def categorize_and_summarize_results(results):
    # Categorize instances based on their pass and pytest results
    categories = {
        'pass_true_pytest_true': [],
        'pass_true_pytest_false': [],
        'pass_false_pytest_true': [],
        'pass_false_pytest_false': []
    }

    # Categorize each instance
    for instance_id, result_data in results.items():
        pass_val = result_data.get('pass', False)
        pytest_val = result_data.get('pytest', False)
        
        if pass_val and pytest_val:
            categories['pass_true_pytest_true'].append(instance_id)
        elif pass_val and not pytest_val:
            categories['pass_true_pytest_false'].append(instance_id)
        elif not pass_val and pytest_val:
            categories['pass_false_pytest_true'].append(instance_id)
        else:
            categories['pass_false_pytest_false'].append(instance_id)

    # Print the count of each category
    for category, instances in categories.items():
        print(f"{category}: {len(instances)}")
        
    # Print pass rate and pytest rate
    total = len(results)
    print(f'num instances: {total}')

    pass_rate = 100 * len(categories['pass_true_pytest_true']) / total
    pytest_rate = 100 * (len(categories['pass_true_pytest_true']) + len(categories['pass_false_pytest_true'])) / total

    print(f"pass rate %: {pass_rate:.2f}")
    print(f"pytest rate %: {pytest_rate:.2f}")
    return pass_rate, pytest_rate
categorize_and_summarize_results(results)


打印一系列模型实验结果

In [ ]:
import pandas as pd
# 创建数据列表
data = []
# for model in ["Qwen3-0.6B-thinking", "Qwen3-1.7B-thinking", "Qwen3-4B-thinking", "Qwen3-8B-thinking", "Qwen3-14B-thinking", "Qwen3-32B-thinking"]:
# for model in ["Qwen3-0.6B-no-thinking", "Qwen3-1.7B-no-thinking", "Qwen3-4B-no-thinking", "Qwen3-8B-no-thinking", "Qwen3-14B-no-thinking", "Qwen3-32B-no-thinking"]:
# for model in ["QwQ-32B-thinking", "gpt-oss-20b-thinking", "gpt-oss-120b", "deepseek-v3.2", "gemini-2.5-flash", "gemini-2.5-pro", "gpt-5-mini", "gpt-5-chat-latest", "claude-sonnet-4-5-20250929"]:
for model in ["Qwen3-0.6B-no-thinking", "Qwen3-0.6B-thinking",
              "Qwen3-1.7B-no-thinking", "Qwen3-1.7B-thinking",
              "Qwen3-4B-no-thinking", "Qwen3-4B-thinking",
              "Qwen3-8B-no-thinking", "Qwen3-8B-thinking",
              "Qwen3-14B-no-thinking", "Qwen3-14B-thinking",
              "Qwen3-32B-no-thinking", "Qwen3-32B-thinking",
              "QwQ-32B-thinking", "gpt-oss-20b-thinking", "gpt-oss-120b", "deepseek-v3.2", "gemini-2.5-flash", "gemini-2.5-pro", "gpt-5-mini", "gpt-5-chat-latest", "claude-sonnet-4-5-20250929"]:
    print(model)
    p=Path(f'logs/run_evaluation/{model}/gold')
    results=collect_test_results(p)
    pass_rate, pytest_rate = categorize_and_summarize_results(results)
    print('#'*50)
    # 添加到数据列表
    data.append({
        'Model': f'{model}',
        'pytest rate': pytest_rate,
        'pass rate': pass_rate,
    })
# 创建DataFrame
df = pd.DataFrame(data)

# 保存为Excel文件
df.to_excel('model_results.xlsx', index=False)

打印instance的pass rate各种通过情况

In [ ]:
def categorize_and_summarize_pass_rate_results(correct_results,complete_results):
    # Categorize instances based on their pass and pytest results
    correct_categories = {
        'pass_true_pytest_true': [],
    }
    complete_categories = {
        'pass_true_pytest_true': [],
    }

    # Categorize each instance
    for instance_id, result_data in correct_results.items():
        pass_val = result_data.get('pass', False)
        pytest_val = result_data.get('pytest', False)
        
        if pass_val and pytest_val:
            correct_categories['pass_true_pytest_true'].append(instance_id)
            
    for instance_id, result_data in complete_results.items():
        pass_val = result_data.get('pass', False)
        pytest_val = result_data.get('pytest', False)
        
        if pass_val and pytest_val:
            complete_categories['pass_true_pytest_true'].append(instance_id)
            
    # Print the count of each category
    for category, instances in correct_categories.items():
        print(f"correct_categories: {len(instances)}")
    for category, instances in complete_categories.items():
        print(f"complete_categories: {len(instances)}")
    # Print pass rate and pytest rate
    total = len(correct_results)
    print(f'num instances: {total}')

    correct_rate = 100 * len(correct_categories['pass_true_pytest_true']) / total
    complete_rate = 100 * len(complete_categories['pass_true_pytest_true']) / total
    pass_rate = 100 * len(set(correct_categories['pass_true_pytest_true']) & set(complete_categories['pass_true_pytest_true'])) / total
    
    print(f"correct_rate %: {correct_rate:.2f}")
    print(f"complete_rate %: {complete_rate:.2f}")
    print(f"pass_rate %: {pass_rate:.2f}")
    return correct_rate, complete_rate, pass_rate

打印一系列模型pass rate实验结果

In [ ]:
import pandas as pd
# 创建数据列表
data = []
# for model in ["Qwen3-0.6B-thinking", "Qwen3-1.7B-thinking", "Qwen3-4B-thinking", "Qwen3-8B-thinking", "Qwen3-14B-thinking", "Qwen3-32B-thinking"]:
# for model in ["Qwen3-0.6B-no-thinking", "Qwen3-1.7B-no-thinking", "Qwen3-4B-no-thinking", "Qwen3-8B-no-thinking", "Qwen3-14B-no-thinking", "Qwen3-32B-no-thinking"]:
# for model in ["QwQ-32B-thinking", "gpt-oss-20b-thinking", "gpt-oss-120b", "deepseek-v3.2", "gemini-2.5-flash", "gemini-2.5-pro", "gpt-5-mini", "gpt-5-chat-latest", "claude-sonnet-4-5-20250929"]:
for model in ["Qwen3-0.6B-no-thinking", "Qwen3-0.6B-thinking",
              "Qwen3-1.7B-no-thinking", "Qwen3-1.7B-thinking",
              "Qwen3-4B-no-thinking", "Qwen3-4B-thinking",
              "Qwen3-8B-no-thinking", "Qwen3-8B-thinking",
              "Qwen3-14B-no-thinking", "Qwen3-14B-thinking",
              "Qwen3-32B-no-thinking", "Qwen3-32B-thinking",
              "QwQ-32B-thinking", "gpt-oss-20b-thinking", "gpt-oss-120b", "deepseek-v3.2", "gemini-2.5-flash", "gemini-2.5-pro", "gpt-5-mini", "gpt-5-chat-latest", "claude-sonnet-4-5-20250929"]:
    print(model)
    correct_results=collect_test_correct_results(Path(f'/mnt/large/SWE-bench-spec/logs/test_correct/{model}/gold'))
    complete_results=collect_test_complete_results(Path(f'/mnt/large/SWE-bench-spec/logs/test_complete/{model}/gold'))
    correct_rate, complete_rate, pass_rate = categorize_and_summarize_pass_rate_results(correct_results,complete_results)
    print('#'*50)
    # 添加到数据列表
    data.append({
        'Model': f'{model}',
        'pass_rate': pass_rate,
        'correct_rate': correct_rate,
        'complete_rate': complete_rate,
    })
# 创建DataFrame
df = pd.DataFrame(data)

# 保存为Excel文件
df.to_excel('model_results.xlsx', index=False)